# 06_xray_file — State x-ray to files + watch_events filtering

Two extensions from `05_xray_console`:

1. **Write to files** — `ConsoleSpanExporter` and `ConsoleLogRecordExporter` accept an `out` parameter (any file-like object). Pass an open file handle to redirect output.

2. **`watch_events`** — filter which event types reach the plugin. By default the plugin receives every event. `watch_events=frozenset({...})` limits it to an explicit set.

This notebook runs **two plugin instances on the same machine**:
- `plugin_traces` — full tracing (all events), writes to `traces.txt`
- `plugin_logs` — only `AfterRegularAspectEvent + GlobalFinishEvent`, writes to `logs.txt`

In [ ]:
!pip install "aoa-action-machine[otel]"

In [ ]:
import asyncio
import os

from opentelemetry.sdk._logs import LoggerProvider
from opentelemetry.sdk._logs.export import ConsoleLogRecordExporter, SimpleLogRecordProcessor
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor
from pydantic import Field

from aoa.action_machine.auth import GuestRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import regular_aspect, summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.checkers import result_float, result_int, result_string
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.plugin.core.events import AfterRegularAspectEvent, GlobalFinishEvent
from aoa.action_machine.plugin.open_telemetry import OpenTelemetryPlugin
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

## Action (same as 05_xray_console)

In [ ]:
class CheckoutDomain(BaseDomain):
    name = "checkout"
    description = "E-commerce checkout pipeline"


class CheckoutParams(BaseParams):
    sku: str = Field(description="Product SKU")
    quantity: int = Field(description="Number of units")
    unit_price: float = Field(description="Price per unit in USD")


class CheckoutResult(BaseResult):
    sku: str = Field(description="Validated SKU")
    total: float = Field(description="Order total in USD")
    confirmation: str = Field(description="Confirmation message")


@meta(description="Three-step checkout: validate → enrich → confirm", domain=CheckoutDomain)
@check_roles(GuestRole)
class CheckoutAction(BaseAction[CheckoutParams, CheckoutResult]):

    @regular_aspect("Step 1: Validate SKU and quantity")
    @result_string("sku", required=True, min_length=3)
    @result_int("quantity", required=True, min_value=1, max_value=500)
    async def validate_aspect(self, params, state, box, connections):
        return {"sku": params.sku.strip().upper(), "quantity": params.quantity}

    @regular_aspect("Step 2: Calculate total")
    @result_string("sku", required=True)
    @result_int("quantity", required=True)
    @result_float("total", required=True, min_value=0.01)
    async def enrich_aspect(self, params, state, box, connections):
        return {"sku": state["sku"], "quantity": state["quantity"],
                "total": round(state["quantity"] * params.unit_price, 2)}

    @summary_aspect("Step 3: Confirm order")
    async def confirm_summary(self, params, state, box, connections):
        return CheckoutResult(
            sku=state["sku"], total=state["total"],
            confirmation=f"Order confirmed: {state['quantity']}x {state['sku']} = ${state['total']}",
        )

## Two plugins on one machine

`plugin_traces` receives all events and writes spans to `traces.txt`.
`plugin_logs` receives only `AfterRegularAspectEvent` and `GlobalFinishEvent`
and writes logs to `logs.txt`.

`watch_events` uses `issubclass` — pass parent types to catch all their subtypes.

In [ ]:
with open("traces.txt", "w") as traces_file, open("logs.txt", "w") as logs_file:

    # Full tracing — all events, spans to file
    tp = TracerProvider()
    tp.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter(out=traces_file)))
    plugin_traces = OpenTelemetryPlugin(tracer_provider=tp, service_name="checkout-service")

    # Focused logs — only state x-ray after each regular step and finish
    lp = LoggerProvider()
    lp.add_log_record_processor(SimpleLogRecordProcessor(ConsoleLogRecordExporter(out=logs_file)))
    plugin_logs = OpenTelemetryPlugin(
        logger_provider=lp,
        service_name="checkout-service",
        watch_events=frozenset({AfterRegularAspectEvent, GlobalFinishEvent}),
    )

    machine = ActionProductMachine(plugins=[plugin_traces, plugin_logs])
    result = await machine.run(
        Context(), CheckoutAction(),
        CheckoutParams(sku=" widget-42 ", quantity=3, unit_price=19.99),
    )

print(f"Result: {result.confirmation}")
print()
print("traces.txt — root span + 2 child spans")
print("logs.txt   — only AfterRegularAspectEvent + GlobalFinishEvent (Before*/Start filtered out)")

## Inspect the output files

In [ ]:
print("=" * 60)
print("logs.txt (filtered — only after-step and finish records):")
print("=" * 60)
with open("logs.txt") as f:
    print(f.read())